In [46]:
import pandas as pd

dados = pd.read_csv('DENGBR25.csv', sep=',')

/tmp/ipython-input-31766338.py:3: DtypeWarning: Columns (44,45,46,54,74,101) have mixed types. Specify dtype option on import or set low_memory=False.
  dados = pd.read_csv('DENGBR25.csv', sep=',')


In [47]:
dados.shape

(1622438, 121)

### Quantidade de registros duplicados.

In [50]:
duplicados = dados[dados.duplicated()]

qtd_duplicados = dados.duplicated().sum()
print(f"Quantidade de registros duplicados: {qtd_duplicados}")

Quantidade de registros duplicados: 649




### Remoção de 649 registros duplicados, que podem distorcer métricas, gerar vieses e prejudicar o modelo.

In [51]:
dados = dados.drop_duplicates()

In [52]:
dados.shape

(1621789, 121)

### Inferência sobre os dados - Avaliando as métricas

- Muitos campos com registros NaN, esses campos serão tratados um a um de acordo com sua particularidade no dicionário de dados. URL: https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SINAN/Dengue/dic_dados_dengue.pdf

- Colunas com 100% de nulos em seus registros serão excluídas.

- Visando obter para análise colunas com ≥ 70% de nulos em seus registros, na maioria dos casos essas colunas não agregam informação preditiva e aumentam o ruído, com exceções clínicas, por exemplo, variáveis de gravidade como: "hospitalizado" ou "obito", exames laboratoriais e sintomas como: "dores", "sangramentos" e "vomitos". Com base nesses critérios, serão feitos análises para remoção das colunas abaixo:


In [53]:
total_linhas = len(dados)

colunas_com_muitos_nulos = []

for coluna in dados.columns:
    porcentagem_nulos = dados[coluna].isna().mean() * 100

    if porcentagem_nulos >= 70:
        colunas_com_muitos_nulos.append(coluna)
        print(f"'{coluna}' - '{porcentagem_nulos:.2f}%'")


'ID_OCUPA_N' - '72.86%'
'DT_CHIK_S1' - '99.93%'
'DT_CHIK_S2' - '100.00%'
'DT_PRNT' - '99.98%'
'RES_CHIKS1' - '99.77%'
'RES_CHIKS2' - '99.79%'
'RESUL_PRNT' - '99.78%'
'DT_SORO' - '83.75%'
'DT_VIRAL' - '99.48%'
'DT_PCR' - '96.09%'
'SOROTIPO' - '97.09%'
'DT_INTERNA' - '95.99%'
'UF' - '95.95%'
'MUNICIPIO' - '96.01%'
'DOENCA_TRA' - '100.00%'
'CLINC_CHIK' - '99.65%'
'DT_OBITO' - '99.85%'
'ALRM_HIPOT' - '97.64%'
'ALRM_PLAQ' - '97.64%'
'ALRM_VOM' - '97.64%'
'ALRM_SANG' - '97.64%'
'ALRM_HEMAT' - '97.65%'
'ALRM_ABDOM' - '97.62%'
'ALRM_LETAR' - '97.64%'
'ALRM_HEPAT' - '97.65%'
'ALRM_LIQ' - '97.65%'
'DT_ALRM' - '97.69%'
'GRAV_PULSO' - '99.81%'
'GRAV_CONV' - '99.81%'
'GRAV_ENCH' - '99.81%'
'GRAV_INSUF' - '99.81%'
'GRAV_TAQUI' - '99.81%'
'GRAV_EXTRE' - '99.81%'
'GRAV_HIPOT' - '99.81%'
'GRAV_HEMAT' - '99.81%'
'GRAV_MELEN' - '99.81%'
'GRAV_METRO' - '99.81%'
'GRAV_SANG' - '99.81%'
'GRAV_AST' - '99.81%'
'GRAV_MIOC' - '99.81%'
'GRAV_CONSC' - '99.81%'
'GRAV_ORGAO' - '99.81%'
'DT_GRAV' - '99.82%'
'MANI_HEM

### Remoção de colunas com valores 100% nulos em seus registros.
  - DT_CHIK_S2 = Data da coleta do soro sorológico (IgM) Chikungunya soro 2.

  - DOENCA_TRA = Indica se a doença está ou não relacionada ao trabalho.

  - MANI_HEMOR = Manifestações hemorrágicas.
  
  - EPISTAXE = Epistaxe.
  
  - GENGIVO = Gengivorrágia.

  - METRO = Metrorrágia.
  
  - PETEQUIAS = Petéquias.
  
  - HEMATURA = Hematúria.

  - SANGRAM = Sangramento Gastrointestinal.

  - LACO_N = Prova do laço positiva.

  - PLASMATICO = Houve Extravasamento Plasmático.

  - EVIDENCIA = Evidenciado por:.
  
  - PLAQ_MENOR = Plaquetas (Menor).

  - CON_FHD = Caso De Fhd/SCD, Especificar.

  - COMPLICA = No Caso De Dengue Com Complicações, Que Tipo De Complicações?.

  - FLXRECEBI = Identifica se o registro foi sistema recebido pelo fluxo de retorno.

  - MIGRADO_W = Identifica se o registro é oriundo da rotina de migração da base Windows

In [54]:
colunas_para_remocao = ['DT_CHIK_S2', 'DOENCA_TRA', 'MANI_HEMOR', 'EPISTAXE', 'GENGIVO', 'METRO', 'PETEQUIAS', 'HEMATURA', 'SANGRAM', 'LACO_N', 'PLASMATICO', 'EVIDENCIA', 'PLAQ_MENOR', 'CON_FHD', 'COMPLICA', 'FLXRECEBI',
'MIGRADO_W']

dados = dados.drop(columns=colunas_para_remocao)

In [55]:
dados.shape

(1621789, 104)

### Remoção de campos com valores acima de 70% de nulos em seus registros, e também por não ter o campo "ID_AGRAVO" igual a A92 ("Febre de Chikungunya")

  - DT_CHIK_S1 = Data da Coleta Exame Sorológico (IgM) Chikungunya soro 1.
  
  - DT_PRNT = Data da Coleta Exame PRNT.
  
  - RES_CHIKS1 = Resultado do Exame Sorológico (IgM) soro 1.
  
  - RES_CHIKS2 = Resultado do Exame Sorológico (IgM) soro 2.
  
  - RESUL_PRNT = Resultado do Exame Sorológico (IgM) PRNT.
  
  - DT_SORO = Data da Coleta Exame Sorológico (IgM) Dengue.
  
  - DT_VIRAL = Data da Coleta Isolamento Viral.
  
  - DT_PCR = Data de Coleta do Exame de RT-PCR.

In [56]:
colunas_para_remocao = ['DT_CHIK_S1', 'DT_PRNT', 'RES_CHIKS1', 'RES_CHIKS2', 'RESUL_PRNT', 'DT_SORO', 'DT_VIRAL', 'DT_PCR']

dados = dados.drop(columns=colunas_para_remocao)

In [57]:
dados.shape

(1621789, 96)

### CAMPOS CATEGÓRICOS (1-SIM / 2-NÃO) - Dengue Grave

  Os registros nulos serão categorizados com o valor 2, conforme dicionário de dados do SUS Dengue.


  - GRAV_INSUF - 99.81% campos nulos' = Dengue grave (Acumulo liquido insuficiencia respiratoria).

  - GRAV_TAQUI - 99.81% = Dengue grave (taquicardia).

  - GRAV_EXTRE - 99.81% = Dengue grave (Extremidade frias).

  - GRAV_HIPOT - 99.81% = Dengue grave (Hipotensao arterial em fase tardia).

  - GRAV_HEMAT - 99.81% = Dengue grave (Hematemese).

  - GRAV_MELEN - 99.81% = Dengue grave (Melena).

  - GRAV_METRO - 99.81% = Dengue grave (Metrorragia volumosa).

  - GRAV_SANG - 99.81% = Dengue grave (Sangramento do SNC).

  - GRAV_AST - 99.81% = Dengue grave (AST/ALT >1.000).

  - GRAV_MIOC - 99.81% = Dengue grave (Miocardite).

  - GRAV_CONSC - '99.81%'	= Dengue grave (Alteração de consciencia).

  - GRAV_ORGAO - '99.81%'	= Dengue grave (Outros orgaos).

In [58]:
colunas_categoricas = ['GRAV_INSUF', 'GRAV_TAQUI', 'GRAV_EXTRE', 'GRAV_HIPOT', 'GRAV_HEMAT', 'GRAV_MELEN', 'GRAV_METRO', 'GRAV_SANG', 'GRAV_AST', 'GRAV_MIOC', 'GRAV_CONSC', 'GRAV_ORGAO']

dados[colunas_categoricas] = dados[colunas_categoricas].fillna(2)

In [59]:
dados.head()

,TP_NOT,ID_AGRAVO,DT_NOTIFIC,SEM_NOT,NU_ANO,SG_UF_NOT,ID_MUNICIP,ID_REGIONA,ID_UNIDADE,DT_SIN_PRI,...,GRAV_SANG,GRAV_AST,GRAV_MIOC,GRAV_CONSC,GRAV_ORGAO,DT_GRAV,TP_SISTEMA,NDUPLIC_N,DT_DIGITA,CS_FLXRET
0,2,A90,2025-11-26,202548,2025,15,150555,1495.0,2615975.0,2025-11-18,...,2.0,2.0,2.0,2.0,2.0,NaN,2.0,NaN,2025-11-28,0.0
1,2,A90,2025-11-26,202548,2025,15,150555,1495.0,2615975.0,2025-11-26,...,2.0,2.0,2.0,2.0,2.0,NaN,2.0,NaN,2025-11-28,0.0
2,2,A90,2025-12-08,202550,2025,15,150555,1495.0,2615975.0,2025-12-07,...,2.0,2.0,2.0,2.0,2.0,NaN,2.0,NaN,2025-12-12,0.0
3,2,A90,2025-12-08,202550,2025,15,150555,1495.0,2615975.0,2025-12-07,...,2.0,2.0,2.0,2.0,2.0,NaN,2.0,NaN,2025-12-12,0.0
4,2,A90,2025-12-08,202550,2025,15,150555,1495.0,2615975.0,2025-12-07,...,2.0,2.0,2.0,2.0,2.0,NaN,2.0,NaN,2025-12-12,0.0


### Arquivo CBO disponivel em:

http://www.mtecbo.gov.br/cbosite/pages/downloads.jsf;jsessionid=KNCtl6LCMUtIVF3HBrrrgdoZpmPmFy9Cn-Hm7qDe.CBO-SLV02:mte-cbo


In [81]:
dados['ID_OCUPA_N'].head(20)


,ID_OCUPA_N
0,0
1,0
2,0
3,0
4,0
5,0
6,0
7,0
8,621005
9,715125


### Adicionado dataset de CBO para fazer o merge com o dataset "DENGBR25"

In [61]:
dados_cbo = pd.read_csv('CBO_Ocupacao.csv', sep=';', encoding='latin1')


### Verifica o tipo do campo "ID_OCUPA_N", para que o merge possa acontecer com o campo "CODIGO" do dataset de CBO.

In [69]:
dados['ID_OCUPA_N'].dtype

dtype('int64')

In [82]:
dados.shape

(1621789, 97)

### Adicionando as duas colunas do arquivo CBO_Ocupacao.csv, visando fazer depara do campo "CODIGO" com o "ID_OCUPA_N" do dataset "DENGBR25.csv", obtendo o campo "TITULO" da ocupação no dataset "DENGBR25".  

In [70]:
dados = dados.merge(
    dados_cbo[['CODIGO', 'TITULO']],
    left_on='ID_OCUPA_N',
    right_on='CODIGO',
    how='left'
)


### Validando o merge tendo a inclusão dos campos "CODIGO" e "TITULO" no dataset "DENGBR25"

In [72]:
dados.shape

(1621789, 98)

In [67]:
dados_cbo.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2719 entries, 0 to 2718
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   CODIGO  2719 non-null   int64 
 1   TITULO  2719 non-null   object
dtypes: int64(1), object(1)
memory usage: 42.6+ KB


### Alterando a coluna ID_OCUPA_N de "object" para "int64"

In [68]:
dados['ID_OCUPA_N'] = pd.to_numeric(dados['ID_OCUPA_N'], errors='coerce')
dados['ID_OCUPA_N'] = dados['ID_OCUPA_N'].fillna(0).astype('int64')

### Validadando alteração.

In [75]:
dados['ID_OCUPA_N'].dtype

dtype('int64')

In [76]:
dados[['ID_OCUPA_N', 'CODIGO', 'TITULO']].head(20)


,ID_OCUPA_N,CODIGO,TITULO
0,0,NaN,NaN
1,0,NaN,NaN
2,0,NaN,NaN
3,0,NaN,NaN
4,0,NaN,NaN
5,0,NaN,NaN
6,0,NaN,NaN
7,0,NaN,NaN
8,621005,621005.0,Trabalhador agropecuário em geral
9,715125,715125.0,Operador de máquinas de construção civil e min...


### Removendo a coluna "CODIGO", coluna não necessária, tendo a "ID_OCUPA_N".

In [78]:
dados.drop(columns=['CODIGO'], inplace=True)

### Validando remoção da coluna "CODIGO"

In [79]:
dados.shape

(1621789, 97)

### Os registros do campo "ID_OCUPA_N" quando nulos serão categorizados com "0"(Zero) e na coluna "TITULO" serão categorizados como "Não informado".

In [86]:
dados['ID_OCUPA_N'] = dados['ID_OCUPA_N'].fillna(0)
dados['TITULO'] = dados['TITULO'].fillna('Não informado')

In [87]:
dados[['ID_OCUPA_N', 'TITULO']].head(20)

,ID_OCUPA_N,TITULO
0,0,Não informado
1,0,Não informado
2,0,Não informado
3,0,Não informado
4,0,Não informado
5,0,Não informado
6,0,Não informado
7,0,Não informado
8,621005,Trabalhador agropecuário em geral
9,715125,Operador de máquinas de construção civil e min...
